# Unit and integration tests

Unit tests check each part of the code on its own, using small synthetic inputs, so they run
in seconds and need no real documents. Integration tests run the real OCR models.

Run this notebook from the `testing-reports` folder, with the project's virtual environment as
the kernel. It takes about 2 minutes.

In [1]:
from report_helpers import *  # noqa: F403  (paths and small display helpers)

## Backend (pytest)

| Area | Examples of what is checked |
|---|---|
| MRZ (machine-readable zone) | TD1, TD2 and TD3 formats; check digits; OCR mix-ups such as 0/O in names |
| Field extraction | Captions above, beside or below values; captions in several languages; names, dates, countries, sex |
| Layout | Side-by-side captions; split captions; relationship lines such as "S/O" under a name |
| Dates | Day/month order, two-digit years, impossible dates, expiry before birth |
| Images | Decoding, size limits, PDF pages, cropping, rotation, brightness |
| OCR recovery | Rereading unclear text; Tesseract fallback; cancellation |
| API and privacy | Sessions, ownership, size and type limits, results deleted after delivery, nothing written to disk |

17 tests are skipped in a normal run on purpose: 16 are integration tests (next section) and
1 needs a private list of authorised sample documents that is not part of the project.

In [2]:
exit_code, output, seconds = run([PY, "-m", "pytest", "-q", "-p", "no:cacheprovider"], BACKEND)
backend = pytest_counts(output)
table(["Passed", "Failed", "Skipped", "Time"],
      [[backend.get("passed", 0), backend.get("failed", 0), backend.get("skipped", 0), f"{seconds:.0f} s"]])
if exit_code:
    print(output[-3000:])

| Passed | Failed | Skipped | Time |
|---|---|---|---|
| 665 | 0 | 17 | 34 s |

## Integration tests (real OCR models)

These are skipped by default because they need the installed model files. Here they are switched
on with `DOCULENS_TEST_REAL_OCR=1`. The count includes the ordinary tests in the same two files.

In [3]:
exit_code, output, seconds = run(
    [PY, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_real_ocr.py",
     "tests/test_visual_label_semantics.py", "-m", "integration or not integration"],
    BACKEND, env={"DOCULENS_TEST_REAL_OCR": "1"})
integration = pytest_counts(output)
table(["Passed", "Failed", "Time"], [[integration.get("passed", 0), integration.get("failed", 0), f"{seconds:.0f} s"]])
if exit_code:
    print(output[-3000:])

| Passed | Failed | Time |
|---|---|---|
| 116 | 0 | 49 s |

## Frontend (Vitest)

Uploading (images and PDFs), refusing unsupported files, scan progress, cancelling, time-outs,
showing results and previews, field highlighting, formatting of dates and countries,
accessibility (keyboard use, labels) and reduced-motion settings.

> **Windows note:** the notebook runs Vitest from a path with an uppercase drive letter. From a
> terminal whose path starts with a lowercase `d:\`, Vitest fails with "failed to find the
> current suite"; `cd D:\...` fixes it.

In [4]:
exit_code, output, seconds = run("npm test", FRONTEND, shell=True)
frontend = vitest_counts(output)
table(["Passed", "Failed", "Time"], [[frontend["passed"], frontend["failed"], f"{seconds:.0f} s"]])
if exit_code:
    print(output[-3000:])

| Passed | Failed | Time |
|---|---|---|
| 87 | 0 | 11 s |

## Code checks

In [5]:
checks = [
    ("Python lint (ruff)", [PY, "-m", "ruff", "check", "app", "tests", "../scripts"], BACKEND, False),
    ("TypeScript type check", "npm run typecheck", FRONTEND, True),
    ("Frontend lint (ESLint)", "npm run lint", FRONTEND, True),
    ("Formatting (Prettier)", "npm run format:check", FRONTEND, True),
    ("Production build (Vite)", "npm run build", FRONTEND, True),
]
rows = []
for name, command, cwd, shell in checks:
    exit_code, output, seconds = run(command, cwd, shell=shell)
    rows.append([name, "✅ clean" if exit_code == 0 else "❌ problems", f"{seconds:.0f} s"])
table(["Check", "Result", "Time"], rows)

| Check | Result | Time |
|---|---|---|
| Python lint (ruff) | ✅ clean | 0 s |
| TypeScript type check | ✅ clean | 3 s |
| Frontend lint (ESLint) | ✅ clean | 2 s |
| Formatting (Prettier) | ✅ clean | 1 s |
| Production build (Vite) | ✅ clean | 5 s |

## Summary

In [6]:
total_passed = backend.get("passed", 0) + frontend["passed"]
total_failed = backend.get("failed", 0) + frontend["failed"] + integration.get("failed", 0)
print(f"Unit tests: {total_passed} passed, {total_failed} failed")
print(f"Integration: {integration.get('passed', 0)} passed")
print(f"Code checks: {sum('✅' in row[1] for row in rows)} of {len(rows)} clean")

Unit tests: 752 passed, 0 failed
Integration: 116 passed
Code checks: 5 of 5 clean
